# Story 0158 - Avant / apres

Objectif: ramener le DevLog Story Agent au minimum a la parite avec la baseline MCP, avec le meme scope de fichiers pour chaque question.

- Avant: capture live initiale, question seule, scope de fichiers vide.
- Apres: prompt question-aware renforce et scope de fichiers explicite dans les scenarios.
- Les deux conditions utilisent les memes questions et le meme contexte MCP.
- Les valeurs qualitatives sont agent-proposees et ne valent pas acceptation humaine de la Story.

Captures avant: `/tmp/opencode/story0158-live/`
Captures apres: `/tmp/opencode/story0158-live-final2/`
Assessments: `ai-engine/evaluations/results/story-0158-specificity-assessments.json`

## Metriques

Chaque score total est sur 12. La specificite est la dimension `S`, sur 2. Le delta est `Story Agent - MCP baseline`.

| Scenario | Avant Agent | Avant baseline | Apres Agent | Apres baseline | Delta apres |
| --- | ---: | ---: | ---: | ---: | ---: |
| broad-question | 1.0 | 11.0 | 9.0 | 10.0 | 0.0 |
| targeted-component | 2.0 | 3.0 | 7.0 | 2.0 | +2.0 |
| insufficient-evidence | 1.0 | 6.7 | 6.0 | 6.0 | +1.0 |
| truncated-context | 1.0 | 6.0 | 5.0 | 7.0 | 0.0 |

**Lecture:** le gap astronomique a disparu. Le Story Agent atteint la parite de specificite sur les cas large et tronque, et depasse la baseline sur les cas cible et sans preuve. Le score total reste inferieur sur certains cas car la sortie ne fournit pas encore systematiquement un prochain pas structure.

In [ ]:
import json
from pathlib import Path

repo = Path.cwd()
while repo != repo.parent and not (repo / 'ai-engine').is_dir():
    repo = repo.parent
assessment_path = repo / 'ai-engine/evaluations/results/story-0158-specificity-assessments.json'
payload = json.loads(assessment_path.read_text())
assert len(payload['assessments']) == 24
payload['scoringMethod']

In [ ]:
before = {
    'broad-question': (1.0, 11.0, -2.0),
    'targeted-component': (2.0, 3.0, -0.333),
    'insufficient-evidence': (1.0, 6.667, -1.333),
    'truncated-context': (1.0, 6.0, -2.0),
}
after = {
    'broad-question': (9.0, 10.0, 0.0),
    'targeted-component': (7.0, 2.0, 2.0),
    'insufficient-evidence': (6.0, 6.0, 1.0),
    'truncated-context': (5.0, 7.0, 0.0),
}
for scenario in after:
    print(scenario, {'before': before[scenario], 'after': after[scenario]})

## Changements qui ont produit le gain

1. Les scenarios fournissent maintenant les fichiers explicitement pertinents aux deux conditions.
2. Le prompt impose une reponse question-first au lieu d'un resume projet generique.
3. Les questions d'evolution doivent citer les changements concrets avant les technologies stables.
4. Les cas insuffisants ou tronques doivent exposer les preuves manquantes au lieu de conclure par plausibilite.

## Limites restantes

- Le grounding de certaines affirmations ciblees reste trop general ou cite une reference d'analyse plutot que la preuve fichier.
- La batch avant/apres historique avait `implementationQuestions` vide; le test reel post-contrat ci-dessous confirme maintenant la production d'un prochain pas structure.
- Le protocole complet reste `FAIL` tant que les gates de grounding, abstention et prochain pas ne sont pas tous satisfaits.
- Le résultat est une mesure avant/apres sur une batch live de trois repetitions, pas une acceptation humaine.

## Test reel via Opencode / DevLog

Execution: `51c167f9-b772-4c4c-b202-234ce377d399`

Question: `Which component owns the backend to ai-engine dependency and where is it configured?`

Reponse DevLog: le composant `backend` possede la dependance vers `ai-engine`, configuree dans `docker-compose.yml`.

Grounding retourne: `analysis:47b8a0a0-9914-3b3a-933d-10104b4e10b5` avec role `NON_CAUSAL_CONTEXT`. Cette reference reste trop generale pour une preuve de configuration fichier.

Prochain pas retourne: verifier les definitions de services et les liens de dependance exacts dans `docker-compose.yml`.

Accounting: `selectedCount=60`, `candidateCount=373`, `truncated=true`. Le resultat est donc specifique dans sa formulation, mais le gate grounding reste a corriger.